In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 1000)

In [ ]:
url = "https://raw.githubusercontent.com/J-SNACKKB/FLIP/9f58a97455d4133669c04b0a723f9d6e49e29eb2/splits/gb1/four_mutations_full_data.csv.zip"

gb1 = pd.read_csv(url)
gb1.columns = [col.strip() for col in gb1.columns]
gb1.columns = gb1.columns.str.replace(' ', '_')
gb1.columns = gb1.columns.str.lower()
gb1 = gb1.drop(columns=["sequence"])
print(gb1.head())

In [ ]:

data = gb1.copy()

data.fillna(0, inplace=True)

data["variants"] = data["variants"].astype(str).str.strip()
data["fitness"] = pd.to_numeric(data["fitness"], errors="coerce")
data["hd"] = data["hd"].astype(int)




data = data.dropna()
data = data[data["variants"].str.len() == 4].copy()

for i in range(4):
    data[f"p{i+1}"] = data["variants"].str[i]





data = data.query("variants != 'VDGV'").copy().reset_index(drop=True)  # Remove the wild-type variant

vcols = ['one_vs_rest', 'one_vs_rest_validation',
       'two_vs_rest', 'two_vs_rest_validation', 'three_vs_rest',
       'three_vs_rest_validation', 'sampled', 'sampled_validation',
       'low_vs_high', 'low_vs_high_validation']

for v in vcols:
    data[v] = pd.to_numeric(data[v], errors="coerce")

print(data.head())
print("Number of observations:", len(data))

In [ ]:
position_columns = ["p1", "p2", "p3", "p4"]

X = pd.get_dummies(
    data[position_columns],
    drop_first=True,
    dtype=float
)

y = data["fitness"].to_numpy()

print("Design matrix:", X.shape)

In [ ]:
model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("R²:", model.score(X, y))

In [ ]:
site_effects = {}

for i in range(1, 5):
    position = f"p{i}"

    amino_acids = sorted(data[position].unique())

    effects = {}

    for aa in amino_acids:
        column = f"{position}_{aa}"

        if column in X.columns:
            effects[aa] = model.coef_[X.columns.get_loc(column)]
        else:
            effects[aa] = 0.0

    site_effects[position] = pd.Series(effects).sort_values(
        ascending=False
    )

for position, effects in site_effects.items():
    print(f"\nPosition {position}")
    print(effects)

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for i, (position, effects) in enumerate(site_effects.items()):
    ax = axes.flat[i]
    effects.sort_values().plot.barh(ax=ax)
    ax.set_title(f"Position {position}")
    ax.set_xlabel("Estimated fitness contribution")

plt.tight_layout()
plt.show()

In [ ]:
data["prediction"] = model.predict(X)
data["residual"] = data["fitness"] - data["prediction"]

for hd in sorted(data["hd"].unique()):
    subset = data[data["hd"] == hd]

    r2 = 1 - (
        np.sum(subset["residual"]**2)
        / np.sum((subset["fitness"] - subset["fitness"].mean())**2)
    )

    print(f"HD={hd}: n={len(subset)}, R²={r2:.4f}")

In [ ]:
wt = "VDGV"

wt_fitness = gb1.loc[
    gb1["variants"] == wt, "fitness"
].iloc[0]

single = data[data["hd"] == 1].copy()

single["position"] = single["variants"].apply(
    lambda s: next(i for i in range(4) if s[i] != wt[i])
)

single["substitution"] = single["variants"].apply(
    lambda s: "".join(
        f"{wt[i]}{i+1}{s[i]}"
        for i in range(4) if s[i] != wt[i]
    )
)

single["fitness_change"] = single["fitness"] - wt_fitness

print(
    single[["variants", "substitution", "fitness", "fitness_change"]]
    .sort_values("fitness_change", ascending=False)
    .to_string(index=False)
)

In [ ]:
interaction_features = {}

for i in range(1, 5):
    for j in range(i + 1, 5):

        cols_i = [c for c in X.columns if c.startswith(f"p{i}_")]
        cols_j = [c for c in X.columns if c.startswith(f"p{j}_")]

        for ci in cols_i:
            for cj in cols_j:
                name = f"{ci}:{cj}"
                interaction_features[name] = X[ci].to_numpy() * X[cj].to_numpy()

X_interactions = pd.DataFrame(interaction_features, index=X.index)

X_pairwise = pd.concat([X, X_interactions], axis=1)

print("One-site features:", X.shape[1])
print("Interaction features:", X_interactions.shape[1])
print("Total features:", X_pairwise.shape[1])

In [ ]:
pairwise_model = LinearRegression()
pairwise_model.fit(X_pairwise, y)

print("One-site R²:", model.score(X, y))
print("Pairwise R²:", pairwise_model.score(X_pairwise, y))

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_pair_train = X_pairwise.loc[X_train.index]
X_pair_test = X_pairwise.loc[X_test.index]

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

In [ ]:

model_one = LinearRegression()
model_one.fit(X_train, y_train)

model_pair = LinearRegression()
model_pair.fit(X_pair_train, y_train)

In [ ]:

pred_one = model_one.predict(X_test)
pred_pair = model_pair.predict(X_pair_test)

results = pd.DataFrame({
    "Model": ["One-site", "Pairwise"],
    "R2": [
        r2_score(y_test, pred_one),
        r2_score(y_test, pred_pair)
    ],
    "RMSE": [
        mean_squared_error(y_test, pred_one) ** 0.5,
        mean_squared_error(y_test, pred_pair) ** 0.5
    ],
    "MAE": [
        mean_absolute_error(y_test, pred_one),
        mean_absolute_error(y_test, pred_pair)
    ]
})

print(results.to_string(index=False))